# Source Code verstehen – AST-Chunks für Python

**Fortsetzung zu:** `01-hf-dataset.ipynb` und `02-hf-dataset-chunks.ipynb`

**Worum geht es?** Bei Text-RAG wird Source Code oft nach einer festen Zeichenzahl aufgeteilt. Das kann mitten in einer Funktion geschehen. Mit dem **Abstract Syntax Tree (AST)** erkennen wir dagegen Python-Klassen, Funktionen und Methoden als zusammengehörige Einheiten.

**Lernziele**
- Python-Source-Code aus der [GitLab-Gruppe Autoshop-MS / App](https://gitlab.com/ch-mc-b/autoshop-ms/app) als reale Datenbasis verwenden.
- **Fixe Text-Chunks** mit **AST-Chunks** vergleichen.
- Fragen erkennen, für die nicht eine einzelne Textstelle, sondern mehrere Definitionen bzw. Aufrufbeziehungen nötig sind.

**Bewusste Begrenzung:** Wir untersuchen ausschliesslich `*.py`. Für Python genügt das Standardmodul `ast`: **kein Tree-sitter, keine Parser-Pakete und kein Hugging Face**. AST liefert statische Aufrufkandidaten – keine zuverlässige Aussage über tatsächliche Laufzeitaufrufe.

## 0. Vorbereitung und Datenquelle

Wir verwenden die vorhandenen Git-Klone in `repos/autoshop-ms-app/`. Sind keine vorhanden, ermittelt das Notebook die Projekte der GitLab-Gruppe und klont eine begrenzte Anzahl schreibgeschützt (`--depth 1`). Die Repositories werden **nicht gestartet** und ihre Dateien **nicht verändert**.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from urllib.request import Request, urlopen
from urllib.parse import quote, urlencode
from urllib.error import HTTPError, URLError
import ast, json, math, os, re, subprocess

GROUP = "ch-mc-b/autoshop-ms/app"
ROOT = Path("repos/autoshop-ms-app")
ROOT.mkdir(parents=True, exist_ok=True)
REPOSITORY_URLS = []         # Optional: URLs einzelner Repositories eintragen
MAX_REPOSITORIES = 12        # Anzahl neu zu klonender Repositories
USE_SSH = False             # Bei privaten Repositories gegebenenfalls True


def gitlab_projects():
    endpoint = "https://gitlab.com/api/v4/groups/" + quote(GROUP, safe="") + "/projects"
    projects = []
    for page in range(1, 11):
        query = urlencode({"per_page": 100, "page": page,
                           "include_subgroups": "true", "with_shared": "false"})
        token = os.environ.get("GITLAB_TOKEN")
        headers = {"PRIVATE-TOKEN": token} if token else {}
        try:
            with urlopen(Request(endpoint + "?" + query, headers=headers), timeout=15) as response:
                batch = json.load(response)
        except (HTTPError, URLError, TimeoutError) as exc:
            print("GitLab-Projektliste nicht verfügbar:", exc)
            return []
        projects.extend(p for p in batch if not p.get("archived"))
        if len(batch) < 100:
            break
    return sorted(projects, key=lambda p: p["path_with_namespace"])


local_repos = sorted(p for p in ROOT.iterdir() if p.is_dir() and (p / ".git").exists())
if local_repos:
    print("Vorhandene Repositories werden verwendet; kein erneutes Klonen.")
else:
    if REPOSITORY_URLS:
        sources = [(url.removesuffix(".git").rstrip("/").rsplit("/", 1)[-1], url)
                   for url in REPOSITORY_URLS]
    else:
        projects = gitlab_projects()
        sources = [(p["path_with_namespace"].removeprefix(GROUP + "/").replace("/", "__"),
                    p["ssh_url_to_repo"] if USE_SSH else p["http_url_to_repo"])
                   for p in projects]
    for name, url in sources[:MAX_REPOSITORIES]:
        target = ROOT / name
        if target.exists():
            continue
        try:
            result = subprocess.run(["git", "clone", "--depth", "1", url, str(target)],
                                    capture_output=True, text=True, timeout=90,
                                    env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
            if result.returncode:
                print(f"Klonen fehlgeschlagen ({name}): {result.stderr[-220:]}")
        except (OSError, subprocess.TimeoutExpired) as exc:
            print(f"Klonen fehlgeschlagen ({name}): {exc}")
    local_repos = sorted(p for p in ROOT.iterdir() if p.is_dir() and (p / ".git").exists())

if not local_repos:
    raise RuntimeError("Keine Git-Repositories gefunden. Prüfe GitLab-Zugriff oder lege lokale Klone unter " + str(ROOT) + " ab.")

revisions = {}
for repo in local_repos:
    result = subprocess.run(["git", "-C", str(repo), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True)
    revisions[repo.name] = result.stdout.strip() or "unbekannt"
    print(f"{repo.name}: {revisions[repo.name]}")

## 1. AST-Chunks erzeugen: Funktionen statt Zeichenblöcke

**Was machen wir?** Wir lesen ausschliesslich Python-Dateien und suchen mit `ast` nach Klassen, Funktionen und Methoden. Jeder Chunk bekommt seinen **Symbolnamen** sowie **Repository, Datei und Zeilenbereich**. Eine Klassenübersicht zeigt zusätzlich ihre Methoden.

**Beobachtung:** Der Parser kennt die Struktur, nicht die Bedeutung des Programms. Deshalb können wir Funktionsgrenzen exakt erkennen, aber dynamische Methodenaufrufe noch nicht sicher auflösen.

Für einen stabilen Notebook-Lauf werden die Anzahl und Grösse der Dateien begrenzt. Fehlerhafte Python-Dateien werden übersprungen – der Kernel soll nicht abbrechen.

In [ ]:
MAX_FILES = 160
MAX_FILES_PER_REPO = 40
MAX_FILE_BYTES = 100_000
MAX_AST_CHUNKS = 1600
MAX_CODE_CHARS = 6500       # Nur Anzeige- und Retrieval-Grenze
SKIP_DIRS = {".git", ".venv", "venv", "__pycache__", "node_modules", "dist",
             "build", "target", ".tox", ".mypy_cache", "site-packages", "vendor"}


def python_files():
    total = 0
    for repo in local_repos:
        per_repo = 0
        for folder, subdirs, files in os.walk(repo):
            subdirs[:] = sorted(d for d in subdirs if d not in SKIP_DIRS)
            for filename in sorted(files):
                if not filename.endswith(".py"):
                    continue
                path = Path(folder) / filename
                try:
                    if path.is_symlink() or path.stat().st_size > MAX_FILE_BYTES:
                        continue
                except OSError:
                    continue
                yield repo, path
                total += 1
                per_repo += 1
                if total >= MAX_FILES or per_repo >= MAX_FILES_PER_REPO:
                    break
            if total >= MAX_FILES or per_repo >= MAX_FILES_PER_REPO:
                break
        if total >= MAX_FILES:
            break


def source_location(node):
    decorators = getattr(node, "decorator_list", [])
    start = min([node.lineno] + [d.lineno for d in decorators])
    return start, getattr(node, "end_lineno", node.lineno)


def call_names(node):
    names = set()
    for entry in ast.walk(node):
        if isinstance(entry, ast.Call):
            target = entry.func
            if isinstance(target, ast.Name):
                names.add(target.id)
            elif isinstance(target, ast.Attribute):
                names.add(target.attr)
        if len(names) >= 60:
            break
    return sorted(names)


def chunks_for_python(repo, path, source):
    tree = ast.parse(source, filename=str(path), type_comments=True)
    lines = source.splitlines()
    chunks = []

    def visit(body, parents=(), enclosing_class=False):
        for node in body:
            if isinstance(node, ast.ClassDef):
                qualname = ".".join((*parents, node.name))
                methods = [n.name for n in node.body
                           if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
                doc = ast.get_docstring(node) or ""
                summary = f"Klasse {qualname}; Methoden: {', '.join(methods) or 'keine'}; Docstring: {doc[:600]}"
                chunks.append({"repo": repo.name, "path": str(path.relative_to(repo)),
                               "symbol": qualname, "kind": "class",
                               "start_line": node.lineno, "end_line": node.end_lineno,
                               "content": summary, "calls": []})
                visit(node.body, (*parents, node.name), enclosing_class=True)
            elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                qualname = ".".join((*parents, node.name))
                first, last = source_location(node)
                code = "\n".join(lines[first-1:last])
                chunks.append({"repo": repo.name, "path": str(path.relative_to(repo)),
                               "symbol": qualname, "kind": "method" if enclosing_class else "function",
                               "start_line": first, "end_line": last,
                               "content": code[:MAX_CODE_CHARS], "truncated": len(code) > MAX_CODE_CHARS,
                               "calls": call_names(node)})
                # Verschachtelte Funktionen separat auffindbar machen.
                visit(node.body, (*parents, node.name), enclosing_class=False)

    visit(tree.body)
    if not chunks:
        chunks.append({"repo": repo.name, "path": str(path.relative_to(repo)),
                       "symbol": "<module>", "kind": "module", "start_line": 1,
                       "end_line": max(len(lines), 1), "content": source[:MAX_CODE_CHARS], "calls": []})
    return chunks


ast_chunks, parsed_files, parse_errors = [], [], []
for number, (repo, path) in enumerate(python_files(), 1):
    relpath = str(path.relative_to(repo))
    try:
        source = path.read_text(encoding="utf-8-sig", errors="replace")
        entries = chunks_for_python(repo, path, source)
        ast_chunks.extend(entries)
        parsed_files.append((repo.name, relpath, source))
    except (SyntaxError, UnicodeError, ValueError, RecursionError, OSError) as exc:
        parse_errors.append((f"{repo.name}/{relpath}", str(exc)[:160]))
    if number % 25 == 0:
        print(f"{number} Dateien geprüft, {len(ast_chunks)} AST-Chunks erstellt")
    if len(ast_chunks) >= MAX_AST_CHUNKS:
        print("Chunk-Limit erreicht – Stichprobe beendet.")
        break

print(f"Python-Dateien: {len(parsed_files)} | AST-Chunks: {len(ast_chunks)} | Parserhinweise: {len(parse_errors)}")
print("Strukturtypen:", dict(Counter(c["kind"] for c in ast_chunks)))
for c in ast_chunks[:8]:
    print(f'{c["repo"]}/{c["path"]}:{c["start_line"]}  {c["kind"]}: {c["symbol"]}')
if parse_errors:
    print("Übersprungene Dateien (Auszug):", parse_errors[:5])
if not ast_chunks:
    raise RuntimeError("Keine Python-AST-Chunks gefunden. Enthalten die geklonten Repositories .py-Dateien? Prüfe ROOT und die Dateigrenzen.")

## 2. Retrieval: feste Text-Chunks gegen AST-Chunks

**Fragestellung:** Finden wir die Implementierung einer Funktion leichter, wenn ihr Code nicht mitten in einer Methode abgeschnitten wird?

Wir vergleichen zwei Varianten **mit derselben einfachen wortbasierten Suche**: einmal starre Textstücke, einmal AST-Einheiten. Die Suche ist bewusst kein Embedding-Modell und noch **kein LLM-basiertes RAG**.

**Worauf achten?** Sind in den Treffern vollständige Funktionsdefinitionen erkennbar? Stimmen Dateipfad und Zeilennummern?

In [ ]:
def fixed_chunks(source, size=1000, overlap=150):
    for offset in range(0, len(source), size-overlap):
        piece = source[offset:offset+size]
        if piece.strip():
            yield offset, piece


text_chunks = []
for repo, path, source in parsed_files:
    for offset, content in fixed_chunks(source):
        text_chunks.append({"repo": repo, "path": path, "symbol": "Textstück",
                            "kind": "text", "start_line": source.count("\n", 0, offset) + 1,
                            "content": content})


def tokenize(text):
    # snake_case und CamelCase behandeln; ohne zusätzliche Python-Pakete.
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", text)
    return re.findall(r"[a-zA-Z][a-zA-Z0-9]*", text.replace("_", " ").lower())


def find_code(question, chunks, k=4):
    # Kleine BM25-artige Wortsuche, identisch für beide Chunk-Varianten.
    docs = [Counter(tokenize(c["symbol"] + " " + c["path"] + " " + c["content"]))
            for c in chunks]
    document_frequency = Counter(word for doc in docs for word in doc)
    n = len(docs)
    avg_length = sum(map(lambda x: sum(x.values()), docs)) / max(n, 1)
    terms = set(tokenize(question))
    ranked = []
    for i, doc in enumerate(docs):
        length = sum(doc.values())
        score = 0.0
        for term in terms:
            if term not in doc:
                continue
            idf = math.log(1 + (n - document_frequency[term] + 0.5) / (document_frequency[term] + 0.5))
            count = doc[term]
            score += idf * (count * 2.2) / (count + 1.2 * (0.25 + 0.75 * length / max(avg_length, 1)))
        ranked.append((score, i))
    return [(chunks[i], round(score, 3)) for score, i in sorted(ranked, reverse=True)[:k]]


def print_hits(hits):
    for chunk, score in hits:
        print(f'{score:6.3f}  {chunk["repo"]}/{chunk["path"]}:{chunk["start_line"]} – {chunk["symbol"]}')


probe = next((c for c in ast_chunks if c["kind"] in ("method", "function")), ast_chunks[0])
first_question = f'Wo ist {probe["symbol"]} implementiert und was macht diese Funktion?'
print("Frage:", first_question)
print("\nFIXE TEXT-CHUNKS:")
print_hits(find_code(first_question, text_chunks))
print("\nAST-CHUNKS:")
print_hits(find_code(first_question, ast_chunks))
print(f"\nIndexumfang: {len(text_chunks)} Text-Chunks, {len(ast_chunks)} AST-Chunks")

## 3. Schwierige Codefragen: Wer ruft wen auf?

**Frage A:** Wo wird eine Funktion definiert? Eine Fundstelle genügt oft.

**Frage B:** Welche Funktionen rufen eine bestimmte Funktion auf, auch **über mehrere Dateien**? Eine Top-4-Trefferliste kann nicht garantieren, alle Aufrufer zu finden: Dafür müssen wir Aufrufe im grösseren Codebestand untersuchen.

**Frage C:** Welche weiteren Funktionen könnten danach aufgerufen werden? Wir nutzen dazu den AST und zeigen **mögliche** Aufrufziele. Gleiche Funktionsnamen, dynamische Aufrufe, Imports und Typauflösung bleiben Unsicherheiten.

> Das ist **keine bewiesene Unmöglichkeit von RAG**. Ein RAG-System mit passendem Index und genügend Retrieval-Schritten kann solche Fragen ebenfalls bearbeiten. Die Übung zeigt die Grenzen einer **einfachen Top-k-Textsuche**.

In [ ]:
definitions = [c for c in ast_chunks if c["kind"] in ("method", "function")]
by_name = defaultdict(list)
for chunk in definitions:
    by_name[chunk["symbol"].split(".")[-1]].append(chunk)


def targets_of(chunk, maximum=10):
    # Namensübereinstimmungen, keine sichere semantische Bindung.
    found, used = [], set()
    for call in chunk["calls"]:
        candidates = [c for c in by_name.get(call, []) if c["repo"] == chunk["repo"]]
        candidates.sort(key=lambda c: (c["path"] != chunk["path"], c["symbol"] != chunk["symbol"]))
        for candidate in candidates[:4]:
            if candidate is chunk:
                continue
            key = (candidate["repo"], candidate["path"], candidate["symbol"], candidate["start_line"])
            if key not in used:
                found.append(candidate)
                used.add(key)
            if len(found) >= maximum:
                return found
    return found


def expand_calls(entry, depth=2, max_chunks=10):
    selected, queue, seen = [], [(entry, 0)], set()
    while queue and len(selected) < max_chunks:
        current, level = queue.pop(0)
        key = (current["repo"], current["path"], current["symbol"], current["start_line"])
        if key in seen:
            continue
        seen.add(key)
        selected.append(current)
        if level < depth:
            queue.extend((candidate, level + 1) for candidate in targets_of(current))
    return selected


if definitions:
    # Bevorzugt eine Funktion mit potenziellen Querverweisen zu anderen Dateien.
    possible_starts = definitions[:300]
    entry = max(possible_starts, key=lambda c: (
        len({t["path"] for t in targets_of(c) if t["path"] != c["path"]}),
        len(targets_of(c)), len(c["calls"])))
    chain = expand_calls(entry)
    print(f'Untersuchte Funktion: {entry["repo"]}/{entry["path"]}:{entry["start_line"]} {entry["symbol"]}')
    print("Statisch erkannte Namen:", entry["calls"][:12])
    print("Mögliche Ziele (max. zwei Stufen):")
    for target in chain[1:]:
        print(f'  {target["repo"]}/{target["path"]}:{target["start_line"]}  {target["symbol"]}')
    cross_file = len({c["path"] for c in chain}) > 1
    print("Potenzielle Querverweise über Dateigrenzen:", "ja" if cross_file else "nicht gefunden")
else:
    entry, chain = ast_chunks[0], [ast_chunks[0]]
    print("Keine Funktionsdefinition erkannt; nur die erste AST-Einheit steht zur Verfügung.")

question = (f'Welche lokalen Funktionen könnte {entry["symbol"]} aufrufen, '
            'wo sind sie definiert und wie verläuft die weitere Aufrufkette?')
plain_hits = find_code(question, text_chunks, k=4)
ast_hits = find_code(question, ast_chunks, k=4)
print("\nMehrdateien-Frage:", question)
print("\nTop 4 der reinen Textsuche:")
print_hits(plain_hits)
print("\nTop 4 der AST-Suche ohne Aufrufexpansion:")
print_hits(ast_hits)
ref_files = {(c["repo"], c["path"]) for c in chain}
hit_files = {(c["repo"], c["path"]) for c, _ in plain_hits}
print(f"\nIm AST-Aufrufkontext: {len(ref_files)} Datei(en); davon in Text-Top-4: {len(ref_files & hit_files)}")

# Ergebnis für 13-source-code-rag.ipynb speichern (ohne erneutes Klonen oder Parsen).
EXPORT = Path("artifacts/source-code-python-ast.json")
EXPORT.parent.mkdir(parents=True, exist_ok=True)
payload = {"version": 1, "group": GROUP, "revisions": revisions,
           "selected_question": question, "entry": entry,
           "text_hits": [{"chunk": c, "score": s} for c, s in plain_hits],
           "ast_hits": [{"chunk": c, "score": s} for c, s in ast_hits],
           "ast_call_context": chain,
           "notes": "Aufrufkanten sind unbestätigte Namenskandidaten, keine Laufzeitbeobachtungen."}
EXPORT.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nGespeichert für Notebook 13:", EXPORT.resolve())

### Ergebnis

- **AST-Chunking** bewahrt Python-Strukturen und ermöglicht die Navigation zu Definitionen.
- **Fixes Text-Retrieval** findet passende Abschnitte, kennt aber keine expliziten Aufrufbeziehungen.
- **AST-Aufrufkandidaten** können weiteren relevanten Code sichtbar machen. Ob ein Aufruf tatsächlich stattfindet, ist dadurch nicht belegt.